En este ejercicio usaremos una API para extraer datos y luego cargarlos a MongoDB

# **Extracción**

In [ ]:
#importar librerias
import requests
import json

In [ ]:
# URL de la API y parámetros de solicitud (si es necesario)
#para mayor información de la API cea https://randomuser.me/documentation

url = 'https://randomuser.me/api/?results=500&nat=es'

response = requests.get(url)

# Verifica si la solicitud fue exitosa (código de respuesta 200)
if response.status_code == 200:
    # Convierte la respuesta a formato JSON (si es JSON)
    data = response.json()
    # Procesa y trabaja con los datos según tus necesidades
    print("Datos de la API:")
    print(json.dumps(data, indent=4))
else:
    print("La solicitud a la API falló. Código de respuesta:", response.status_code)

# **Transformación**

In [ ]:
#Tener en cuenta que esta ubicación es solo referencial por ser datos aleatorios

def generate_google_maps_link(latitude, longitude):
    """
    Genera un enlace a Google Maps con la ubicación especificada por latitude y longitude.
    """
    return f"https://www.google.com/maps?q={latitude},{longitude}"

In [ ]:
# Extrae y guarda solo los campos que necesitas
filtered_data = []
for result in data['results']:
    filtered_result = {
        "id": result["id"]["name"]+"-"+result["id"]["value"],
        "phone": result["phone"],
        "email": result["email"],
        "first_name": result["name"]["first"],
        "picture": result["picture"]["large"],
        "city": result["location"]["city"],
        "country": result["location"]["country"],
        "maps": generate_google_maps_link(result["location"]["coordinates"]["latitude"],result["location"]["coordinates"]["longitude"])
    }
    filtered_data.append(filtered_result)

In [ ]:
print(filtered_data)

# **CARGA DE DATOS**

In [ ]:
#capturamos la IP de Colab para registrar en la configuración de red de Mongo
!curl ipecho.net/plain

In [ ]:
!pip install pymongo

In [ ]:
#importando la libreria
import pymongo
from pymongo import MongoClient

usuario='alumno'
password='alumno'
server='cluster0.18giq.mongodb.net'
#Creando la cadena de conexión con mongo
client = MongoClient('mongodb+srv://'+usuario+':'+password+'@'+server+'/test?retryWrites=true&w=majority')

In [ ]:
#Validando
#Creamos la BD y collection
db = client['bdcomercial']
collection = db['personas']

In [ ]:
#Insertamos los registro
result = collection.insert_many(filtered_data)

# Imprime la información sobre el resultado de la operación
print("Información sobre el resultado de la inserción:")
print("Número de documentos insertados:", len(result.inserted_ids))
print("IDs de los documentos insertados:", result.inserted_ids)



In [ ]:
#leemos los registros
person_mongo = collection.find()
for documentoPerson in person_mongo:
    print(documentoPerson)